Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 3/6: Bi-encoder vs cross-encoder: когда нужен reranking]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/29e6c812-9068-4542-b4bf-207c48326728/#8a6510a7-0d54-4288-bd15-ca47bdb46b45 )

In [1]:
from sentence_transformers import CrossEncoder

# Поднимаем реранкер на GPU; device='cpu' — если GPU нет
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda")

query = "какая гарантия на холодильник"
candidates = [
    "на крупную бытовую технику гарантия 24 месяца",
    "колл-центр работает с 8 до 22",
    "гарантийный талон храните вместе с чеком",
]

# predict() принимает список пар (запрос, документ)
# и возвращает по числу-оценке релевантности на каждую пару
scores = reranker.predict([(query, doc) for doc in candidates])
print(scores)  # Например: [6.84 -7.12 2.05] — чем больше, тем релевантнее

[7.1190840e-01 2.4074008e-05 1.7101280e-04]


In [1]:
import numpy as np                                         # Векторы и матричные операции
from collections import defaultdict                        # Группировка разметки по запросам
from datasets import load_dataset                          # Скачивание корпуса MIRACL
from sentence_transformers import SentenceTransformer, CrossEncoder  # Эмбеддер и реранкер

N_QUERIES = 15          # Сколько запросов берём
CORPUS_SIZE = 500       # Размер замкнутого корпуса
MIN_REL = 3             # Берём запросы минимум с тремя релевантными документами

# --- Сборка корпуса из размеченного среза MIRACL ---
# ru-queries: запросы; ru-qrels: разметка релевантности (score 1 = релевантно);
# ru-corpus: тексты документов-кандидатов
queries_ds = load_dataset("mteb/MIRACLReranking", "ru-queries", split="dev")
qrels_ds = load_dataset("mteb/MIRACLReranking", "ru-qrels", split="dev")
corpus_ds = load_dataset("mteb/MIRACLReranking", "ru-corpus", split="dev")

# Разметка: query-id -> {corpus-id: score}; тексты запросов и документов по id
judged = defaultdict(dict)
for r in qrels_ds:
    judged[r["query-id"]][r["corpus-id"]] = r["score"]
qtext = {r["_id"]: r["text"] for r in queries_ds}
ctext = {r["_id"]: r["text"][:1000] for r in corpus_ds}     # Длинные тексты подрезаем

# Берём первые N_QUERIES запросов минимум с MIN_REL релевантными документами
# Несколько релевантных на запрос дают precision@5 запас, чтобы реранкинг было
# видно: на запросе с единственным релевантным метрика упёрлась бы в потолок 0.2
chosen = []
for qid in qtext:
    if sum(1 for s in judged[qid].values() if s >= 1) >= MIN_REL:
        chosen.append(qid)
    if len(chosen) == N_QUERIES:
        break

# Корпус: сперва кладём все релевантные документы выбранных запросов
# (чтобы они точно были внутри), затем добиваем остальными кандидатами до CORPUS_SIZE
corpus_ids, seen = [], set()
for qid in chosen:
    for cid, s in judged[qid].items():
        if s >= 1 and cid not in seen and cid in ctext:
            seen.add(cid); corpus_ids.append(cid)
for qid in chosen:
    for cid in judged[qid]:
        if len(corpus_ids) >= CORPUS_SIZE:
            break
        if cid not in seen and cid in ctext:
            seen.add(cid); corpus_ids.append(cid)
    if len(corpus_ids) >= CORPUS_SIZE:
        break
corpus_ids = corpus_ids[:CORPUS_SIZE]

CORPUS = [ctext[cid] for cid in corpus_ids]
id2idx = {cid: i for i, cid in enumerate(corpus_ids)}
QUERIES = [qtext[qid] for qid in chosen]
RELEVANT = {qi: {id2idx[cid] for cid, s in judged[qid].items()
                 if s >= 1 and cid in id2idx}
            for qi, qid in enumerate(chosen)}
assert len(CORPUS) == CORPUS_SIZE and len(QUERIES) == N_QUERIES
print(f"Корпус: {len(CORPUS)} документов, запросов: {len(QUERIES)}")


# --- Модели: bi-encoder и cross-encoder на GPU ---
bi = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda")

# Документы кодируем заранее (это и есть предвычисление bi-encoder), запросы — тоже
doc_vecs = bi.encode(["passage: " + d for d in CORPUS],
                     normalize_embeddings=True, show_progress_bar=False)
query_vecs = bi.encode(["query: " + q for q in QUERIES],
                       normalize_embeddings=True, show_progress_bar=False)


def retrieve(query_vec, doc_vecs, n=20):
    """Ступень 1: bi-encoder отбирает top-n кандидатов по косинусному сходству.
    Возвращает индексы кандидатов в CORPUS, отсортированные по убыванию сходства."""
    sims = query_vec @ doc_vecs.T          # cos sim для нормализованных векторов
    # Пропуск 1: индексы n документов с наибольшим сходством (по убыванию)
    return np.argsort(sims)[-n:][::-1]


def rerank(query, cand_idx, corpus, reranker, k=5):
    """Ступень 2: cross-encoder переоценивает кандидатов и возвращает top-k.
    query — текст запроса, cand_idx — индексы кандидатов из retrieve()."""
    pairs = [(query, corpus[i]) for i in cand_idx]
    # Пропуск 2: оценки релевантности пар (запрос, документ) от реранкера
    scores = reranker.predict(pairs)
    # Пропуск 3: индексы cand_idx, переупорядоченные по убыванию scores, первые k
    order = [cand_idx[j] for j in np.argsort(scores)[::-1][:k]]
    return order


# --- Прогон по трём первым запросам ---
for qi in range(3):
    query = QUERIES[qi]
    cand = retrieve(query_vecs[qi], doc_vecs, n=20)        # top-20 от bi-encoder
    reranked = rerank(query, cand, CORPUS, reranker, k=5)  # top-5 после реранкера
    rel = RELEVANT[qi]
    print(f"\nЗапрос: {query}")
    print(f"  Биэнкодер top-5 (до реранкинга):")
    for rank, idx in enumerate(cand[:5], 1):
        mark = "+" if idx in rel else " "
        print(f"    {mark} {rank}. #{idx:03d} {CORPUS[idx][:70]}...")
    print(f"  После реранкинга top-5:")
    for rank, idx in enumerate(reranked, 1):
        mark = "+" if idx in rel else " "
        print(f"    {mark} {rank}. #{idx:03d} {CORPUS[idx][:70]}...")

Корпус: 500 документов, запросов: 15

Запрос: Как называлась Симбирская губерния с 1924?
  Биэнкодер top-5 (до реранкинга):
    + 1. #001 Симбирская губерния Симби́рская губе́рния (с 1924 года Ульяновская губ...
    + 2. #002 Симбирская губерния Декретом ЦИК СССР от 9 мая 1924 года город Симбирс...
      3. #072 Симбирский уезд Симбирский уезд (до 1780 г. — Синбирский уезд) — админ...
      4. #088 Симбирская губерния К началу XX века Симбирская губерния занимала 49,5...
    + 5. #000 Список руководителей симбирско-ульяновского региона В 1924 году Симбир...
  После реранкинга top-5:
    + 1. #002 Симбирская губерния Декретом ЦИК СССР от 9 мая 1924 года город Симбирс...
    + 2. #001 Симбирская губерния Симби́рская губе́рния (с 1924 года Ульяновская губ...
    + 3. #000 Список руководителей симбирско-ульяновского региона В 1924 году Симбир...
    + 4. #003 Ульяновск 9 мая 1924 года, в связи со смертью Владимира Ильича Ульянов...
      5. #081 История Ульяновской области В 1924 году Симб

In [2]:
import time
import numpy as np

# CORPUS, QUERIES, RELEVANT, doc_vecs, query_vecs, bi, reranker,
#  retrieve(), rerank() — из Задания 4

N_CAND, K = 20, 5


def precision_at_k(top_indices, relevant_set, k=K):
    """Доля релевантных документов в первых k результатах.
    top_indices — индексы выдачи, relevant_set — множество релевантных."""
    hits = set(int(i) for i in top_indices) & relevant_set
    # Пропуск 1: precision@k = |пересечение| / k
    return len(hits) / k


p_single, p_two = [], []        # precision@5 по запросам для двух подходов
t_single, t_two = 0.0, 0.0      # Суммарная latency по запросам, секунды

for qi, query in enumerate(QUERIES):
    rel = RELEVANT[qi]

    # --- Одноэтапный: bi-encoder кодирует запрос и ищет top-N ---
    t0 = time.perf_counter()
    qv = bi.encode("query: " + query, normalize_embeddings=True)
    cand = retrieve(qv, doc_vecs, n=N_CAND)
    # Пропуск 2: первые K кандидатов bi-encoder как финальная выдача
    single_top = cand[:K]
    t_single += time.perf_counter() - t0

    # --- Двухэтапный: те же отбор и кандидаты, плюс реранкинг ---
    t0 = time.perf_counter()
    qv = bi.encode("query: " + query, normalize_embeddings=True)
    cand = retrieve(qv, doc_vecs, n=N_CAND)
    # Пропуск 3: top-K после реранкинга кандидатов
    two_top = rerank(query, cand, CORPUS, reranker, k=K)
    t_two += time.perf_counter() - t0

    p_single.append(precision_at_k(single_top, rel))
    p_two.append(precision_at_k(two_top, rel))

n = len(QUERIES)
print(f"{'Подход':<30}{'precision@5':>13}{'Латенси/запрос, мс':>22}")
print(f"{'Биэнкодер (одноэтапный)':<30}{np.mean(p_single):>13.3f}"
      f"{t_single / n * 1000:>22.1f}")
print(f"{'Биэнкодер + реранкер':<30}{np.mean(p_two):>13.3f}"
      f"{t_two / n * 1000:>22.1f}")

Подход                          precision@5    Латенси/запрос, мс
Биэнкодер (одноэтапный)               0.707                  22.3
Биэнкодер + реранкер                  0.760                 997.7
